In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import torch

print(torch.cuda.is_available())

True


In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)


model_id = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
lora_adapter_path = "Moris258/Meta-Llama-3.1-8B-Instruct-Manga-Synopsis3-LORA"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", trust_remote_code=True)
lora_loaded_model = PeftModel.from_pretrained(model, lora_adapter_path, device_map="auto", trust_remote_code=True)
pipe = pipeline(
    "text-generation",
    model=lora_loaded_model,
    tokenizer=tokenizer,
    model_kwargs={
        "quantization_config": quantization_config,
    }
)

hf = HuggingFacePipeline(pipeline=pipe)
from langchain_huggingface import ChatHuggingFace
from langchain.messages import HumanMessage, SystemMessage

lora_model = ChatHuggingFace(llm=hf)

W0411 20:45:53.685000 9444 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
prompt_lora = "A young man's journey through turmoil and transformation in feudal Japan."

messages = [SystemMessage(content="You are a helpful assistant that creates manga synopses based on the given description."), HumanMessage(content=prompt_lora)]

response = lora_model.invoke(messages)

print(response.content)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant that creates manga synopses based on the given description.<|eot_id|><|start_header_id|>user<|end_header_id|>

A young man's journey through turmoil and transformation in feudal Japan.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

synopsis: In the year 1603, a revolution took place in Japan that changed the country's course in history. During this revolution, a young man, Ishida Mitsunari, played a crucial role in changing the course of history. This is his story.


In [8]:

from langchain.agents import create_agent

agent = create_agent(
    model = lora_model,
    system_prompt="You are a helpful assistant that creates manga synopses based on the given description.",
)

response = agent.invoke({
    "messages": [HumanMessage(content=prompt_lora)]
})

print(response)


d:\Python\story_gen\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'messages': [HumanMessage(content="A young man's journey through turmoil and transformation in feudal Japan.", additional_kwargs={}, response_metadata={}, id='dee2ec67-ba2f-470f-bcba-a05c996f014c'), AIMessage(content="<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a helpful assistant that creates manga synopses based on the given description.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nA young man's journey through turmoil and transformation in feudal Japan.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nsynopsis: The story is set in the Sengoku period and follows the life of a supposed demon child named Hiryu, who was abandoned by his family and left to fend for himself. The story begins with Hiryu as a baby, and follows his journey as he grows up in the Sengoku period and becomes a mighty warrior.", additional_kwargs={}, response_metadata={}, id='lc_run--019d7de5-4274-7061-a

In [14]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that creates manga synopses based on the given description. The generated synopsis should be about 500 characters."},
    {"role": "user", "content": "A young man's journey through turmoil and transformation in feudal Japan."},
]

outputs = pipe(
    messages,
)

print(outputs[0]["generated_text"][-1]["content"])

d:\Python\story_gen\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


synopsis: The story is set in the Sengoku period and spans the course of the life of Musashi Miyamoto, one of Japan's most famous swordsmen. It begins with Miyamoto as a young boy, and then continues with his growth as a swordsman. It concludes with his famous duel with Sasaki Kojirou.


In [20]:
print(outputs[0]["generated_text"][-1]["content"][10:])

The story is set in the Sengoku period and spans the course of the life of Musashi Miyamoto, one of Japan's most famous swordsmen. It begins with Miyamoto as a young boy, and then continues with his growth as a swordsman. It concludes with his famous duel with Sasaki Kojirou.
